# Module 1: Deep Agents — Internal HR Assistant

> Part of the **Modular Workshops** series. Standalone, ~45 min.

> ⚠️ **All data in this notebook is fully synthetic.** The HR database — employees, salaries, comp bands, reporting lines, and cases — is generated by a seed script (`utils/hr_seed.py`). There is no real employee or personal information anywhere in here.

We build an **internal HR assistant**: an agent that answers employee and manager questions by querying a transactional HR database and then taking an action. It is a **retrieve-then-act** agent (look facts up, then open an HR case) — *not* a RAG-over-documents agent.

Deep Agents = `create_agent()` + a pre-built middleware stack (filesystem, planning, subagents, context management). We'll build up from a bare agent to a fully-featured HR assistant, exploring:

- The harness and built-in tools
- Custom tools (the HR database)
- Subagents and context isolation
- Backends and persistent memory
- Middleware (compliance, audit)
- Human-in-the-loop on tool calls (gating the "act" step)
- AGENTS.md and Skills

## Setup

First we seed the synthetic HR database. This creates `hr.db` (SQLite) with four tables — `employees`, `job_families`, `comp_bands`, and `cases` — and prints a summary that highlights the **deliberate edge cases** we'll use later:

- an employee paid **below their band minimum**
- a **manager with no direct reports**
- a **duplicate name** shared by two employees (lookup by name is ambiguous)
- an employee with a **null salary** (cannot be assessed against band)

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model

from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from langsmith import uuid7
from IPython.display import Image, display

# Seed the fully-synthetic HR database (ALL DATA IS FAKE) at the repo root, so
# the notebook and the shared agents/hr_agent.py factory use the same hr.db.
from utils.hr_seed import seed_hr_db, _summary, DB_PATH
db_path = seed_hr_db(project_root / DB_PATH)
print(_summary(db_path))

# Start each run with a clean long-term memory store (section 1.4 recreates it).
for f in Path().glob("deep_agents_memory.db*"):
    f.unlink()

print("\nReady")

---
# Part 1: Deep Agents

Deep Agents = `create_agent()` + a pre-built middleware stack (filesystem, planning, subagents, context management).

We'll build up from a bare agent to a fully-featured HR assistant.

## 1.1 Your First Deep Agent

`create_deep_agent()` gives you a filesystem, a todo list, and context management out of the box — no tools required.

<img src="../images/deepAgentsDiag.png" style="width: auto; max-height: 420px; border-radius: 8px;">

### What you get for free:

- **Filesystem Tools** — `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`
- **Planning Tool** — `write_todos` for task tracking
- **Subagent Delegation** — `task()` tool for isolated work
- **Large Tool Result Eviction** — Automatically offloads tool results >20k tokens to the filesystem
- **Conversation Summarization** — Compresses history when approaching ~85% context capacity
- **Dangling Tool Call Patching** — Fixes message history consistency automatically

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    system_prompt="You are a helpful internal HR assistant.",
    checkpointer=MemorySaver(),
)
agent

In [ ]:
# The agent can already write and read files — these are built-in tools.
# Here it drafts a short onboarding note to a scratch file, then reads it back.
config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Draft a 3-bullet first-week onboarding checklist for a new hire and save it to /onboarding.md, then read it back to me."}]
}, config=config)

for m in result["messages"]:
    m.model_copy(update={"content": m.text}).pretty_print()

In [ ]:
# Helper: print the virtual filesystem from a deep agent result.
def print_files(result, header="VIRTUAL FILESYSTEM (in-memory, not on disk!)"):
    files = result.get("files") or {}
    if not files:
        print("(no files in state)")
        return
    print("=" * 50)
    print(header)
    print("=" * 50)
    for path, file_data in files.items():
        print(f"\n  Path: {path!r}")
        print("  " + "-" * 38)
        content = file_data
        if isinstance(file_data, dict) and "content" in file_data:
            content = file_data["content"]
        if isinstance(content, list):
            content = "\n".join(content)
        for line in str(content).split("\n"):
            print(f"  | {line}")

print_files(result)

### Filesystem persistence within a thread

By default, `create_deep_agent()` uses **StateBackend** — files are stored in agent state and persist within a thread (via the checkpointer), but disappear when you start a new thread.

| Backend | Storage | Persistence | Use Case |
|---------|---------|-------------|----------|
| **StateBackend** | In-memory (agent state) | Single thread | Scratch pads, intermediate results |
| **FilesystemBackend** | Local disk | Permanent | Direct file access (use with caution) |
| **StoreBackend** | LangGraph Store | Cross-thread | Long-term memories |
| **CompositeBackend** | Routes to others | Mixed | Selective persistence |

In [ ]:
# Same thread — the file persists via the checkpointer
result = agent.invoke({
    "messages": [{"role": "user", "content": "Read the file /onboarding.md"}]
}, config=config)

print("Same thread:\n\n", result["messages"][-1].text)

In [ ]:
# New thread — StateBackend is ephemeral, so the file is gone
new_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "List all files with ls /"}]
}, config=new_config)

print("New thread:", result["messages"][-1].text)

### Key Takeaway
- `create_deep_agent()` gives you filesystem + planning capabilities for free
- Files are stored in agent state (virtual, not on disk)
- `StateBackend` (default) persists within a thread but is ephemeral across threads
- We'll see how to make files persist across threads with `CompositeBackend` + `StoreBackend` in section 1.4

## 1.2 Custom Tools: the HR Database

Now we make the agent useful by giving it tools that query the synthetic HR database. This is what turns a generic assistant into a **retrieve-then-act** HR agent.

We define four tools with `@tool` so the pattern stays visible. They're thin wrappers over SQLite:

| Tool | Retrieve / Act | Purpose |
|------|----------------|---------|
| `lookup_employee(employee_id \| name)` | retrieve | role, department, manager, location, tenure, salary band, salary |
| `list_reports(manager_id)` | retrieve | a manager's direct reports |
| `get_comp_bands(job_family, level)` | retrieve | band min / mid / max |
| `open_hr_case(employee_id, category, summary)` | **act** | writes a row to the `cases` table, returns a case id |

> The same four tools plus a `build_hr_agent()` factory are packaged in `agents/hr_agent.py` — Module 3 imports it to generate traces. Here we inline them to show the pattern.

In [ ]:
import sqlite3
from datetime import date

DB = str(project_root / DB_PATH)

def _connect():
    conn = sqlite3.connect(DB)
    conn.row_factory = sqlite3.Row
    return conn

def _tenure_years(hire_date: str) -> float:
    return round((date(2025, 6, 1) - date.fromisoformat(hire_date)).days / 365.25, 1)


@tool(parse_docstring=True)
def lookup_employee(employee_id: int | None = None, name: str | None = None) -> str:
    """Look up an employee by ID or name.

    Returns role, department, manager, location, tenure, salary band, and salary.
    Prefer employee_id — names may be ambiguous. If a name matches multiple people,
    all matches are returned so you can disambiguate by employee_id.

    Args:
        employee_id: The employee's numeric ID (most reliable).
        name: The employee's full name (may be ambiguous).
    """
    if employee_id is None and not name:
        return "Error: provide either employee_id or name."
    conn = _connect()
    try:
        if employee_id is not None:
            rows = conn.execute("SELECT * FROM employees WHERE employee_id = ?", (employee_id,)).fetchall()
        else:
            rows = conn.execute("SELECT * FROM employees WHERE name = ? COLLATE NOCASE", (name,)).fetchall()
        if not rows:
            who = f"id={employee_id}" if employee_id is not None else f"name={name!r}"
            return f"No employee found for {who}."
        if len(rows) > 1:
            lines = [f"Ambiguous — {len(rows)} employees match {name!r}. Disambiguate by employee_id:"]
            for r in rows:
                lines.append(f"  - id={r['employee_id']}, {r['role']}, {r['department']}, {r['location']}")
            return "\n".join(lines)
        r = rows[0]
        mgr = None
        if r["manager_id"] is not None:
            m = conn.execute("SELECT name FROM employees WHERE employee_id = ?", (r["manager_id"],)).fetchone()
            mgr = f"{m['name']} (id={r['manager_id']})" if m else f"id={r['manager_id']}"
        band = conn.execute(
            "SELECT band_min, band_mid, band_max FROM comp_bands WHERE job_family = ? AND level = ?",
            (r["job_family"], r["level"]),
        ).fetchone()
        band_str = f"{band['band_min']:,}/{band['band_mid']:,}/{band['band_max']:,}" if band else "unknown"
        salary_str = f"${r['salary']:,}" if r["salary"] is not None else "NULL (not on record)"
        return (
            f"employee_id: {r['employee_id']}\n"
            f"name: {r['name']}\n"
            f"role: {r['role']}\n"
            f"department: {r['department']}\n"
            f"job_family / level: {r['job_family']} / {r['level']}\n"
            f"manager: {mgr or 'none'}\n"
            f"location: {r['location']}\n"
            f"tenure: {_tenure_years(r['hire_date'])} years (hired {r['hire_date']})\n"
            f"salary_band (min/mid/max): {band_str}\n"
            f"salary: {salary_str}"
        )
    finally:
        conn.close()

In [ ]:
@tool(parse_docstring=True)
def list_reports(manager_id: int) -> str:
    """List the direct reports of a manager.

    Args:
        manager_id: The manager's employee ID.
    """
    conn = _connect()
    try:
        mgr = conn.execute("SELECT name FROM employees WHERE employee_id = ?", (manager_id,)).fetchone()
        if mgr is None:
            return f"No employee found for manager id={manager_id}."
        rows = conn.execute(
            "SELECT employee_id, name, role, job_family, level, salary "
            "FROM employees WHERE manager_id = ? ORDER BY employee_id",
            (manager_id,),
        ).fetchall()
        if not rows:
            return f"{mgr['name']} (id={manager_id}) has no direct reports."
        lines = [f"{mgr['name']} (id={manager_id}) has {len(rows)} direct report(s):"]
        for r in rows:
            salary_str = f"${r['salary']:,}" if r["salary"] is not None else "NULL"
            lines.append(
                f"  - id={r['employee_id']}, {r['name']}, {r['role']} "
                f"({r['job_family']} {r['level']}), salary {salary_str}"
            )
        return "\n".join(lines)
    finally:
        conn.close()


@tool(parse_docstring=True)
def get_comp_bands(job_family: str, level: str) -> str:
    """Get the compensation band (min / mid / max) for a job family and level.

    Args:
        job_family: e.g. "Software Engineering", "Sales", "Finance".
        level: One of L1, L2, L3, L4, L5.
    """
    conn = _connect()
    try:
        row = conn.execute(
            "SELECT band_min, band_mid, band_max FROM comp_bands "
            "WHERE job_family = ? COLLATE NOCASE AND level = ? COLLATE NOCASE",
            (job_family, level),
        ).fetchone()
        if row is None:
            families = [r["job_family"] for r in conn.execute(
                "SELECT DISTINCT job_family FROM comp_bands ORDER BY job_family").fetchall()]
            return (f"No band for job_family={job_family!r}, level={level!r}. "
                    f"Known families: {', '.join(families)}. Levels: L1-L5.")
        return (f"{job_family} {level} band: "
                f"min ${row['band_min']:,} / mid ${row['band_mid']:,} / max ${row['band_max']:,}")
    finally:
        conn.close()


@tool(parse_docstring=True)
def open_hr_case(employee_id: int, category: str, summary: str) -> str:
    """Open an HR case for an employee (the "act" step — writes to the cases table).

    Use this only after you have looked up the relevant facts. Returns the new case id.

    Args:
        employee_id: The employee the case is about.
        category: Short category, e.g. "comp_review", "band_adjustment", "data_quality".
        summary: One or two sentences describing why the case is being opened.
    """
    conn = _connect()
    try:
        emp = conn.execute("SELECT name FROM employees WHERE employee_id = ?", (employee_id,)).fetchone()
        if emp is None:
            return f"Cannot open case: no employee with id={employee_id}."
        cur = conn.execute(
            "INSERT INTO cases (employee_id, category, summary, status, created_at) "
            "VALUES (?, ?, ?, 'open', ?)",
            (employee_id, category, summary, date(2025, 6, 1).isoformat()),
        )
        conn.commit()
        return f"Opened case #{cur.lastrowid} ({category}) for {emp['name']} (id={employee_id}), status=open."
    finally:
        conn.close()


HR_TOOLS = [lookup_employee, list_reports, get_comp_bands, open_hr_case]

Now build the HR agent with these four tools and ask it a representative question — *"is this employee paid inside band for their level?"* This is the core retrieve-then-act loop: the agent looks up the employee, reads the band, and reasons about where the salary falls.

In [ ]:
hr_system_prompt = (
    "You are an internal HR assistant. You answer employee and manager questions by "
    "querying the HR database, then take an action when asked.\n\n"
    "Guidelines:\n"
    "- Always look up facts with the tools before answering; never guess salaries, "
    "bands, or reporting lines.\n"
    "- Prefer employee_id over name. If a name matches multiple employees, ask the "
    "user to disambiguate rather than picking one.\n"
    "- When checking whether someone is 'inside band', compare their salary to the "
    "band min/mid/max for their job_family and level. If salary is NULL, say it "
    "cannot be assessed.\n"
    "- Only open an HR case (open_hr_case) when the user explicitly asks you to act, "
    "and only after you've confirmed the relevant facts."
)

agent = create_deep_agent(
    model=model,
    tools=HR_TOOLS,
    system_prompt=hr_system_prompt,
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Is employee 1001 paid inside band for their level?"}]
}, config=config)

for m in result["messages"]:
    m.model_copy(update={"content": m.text}).pretty_print()

Employee 1001 is one of our **deliberate edge cases** — paid below their band minimum. The agent should catch that.

Try the other representative questions too (each is a good demo of retrieve-then-act):
- *"Who reports to manager 1201?"* — a manager with **no direct reports**.
- *"Look up Lucia Brown"* (or whichever name the summary flagged) — a **duplicate name**; the agent should ask you to disambiguate.
- *"Is employee 1004 inside band?"* — a **null salary**; cannot be assessed.

## 1.3 Subagents: Isolated Delegation

Subagents run in a separate context. The main agent delegates via `task()` and only sees the final result — keeping the main context clean.

A good HR use case: a **comp-analyst** subagent that reviews an entire team against band. Pulling every report's salary and band into the *main* conversation would bloat its context; instead we delegate the whole "list the reports, check each against band, summarize who's below band mid" job and get back one tidy answer.

<img src="../images/deepAgentSubagents.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
comp_analyst_subagent = {
    "name": "comp-analyst",
    "description": (
        "Delegate compensation reviews of a whole team. Give one manager_id at a time. "
        "The subagent lists the reports, checks each against their band, and returns a "
        "summary of who is below band mid or outside band."
    ),
    "system_prompt": (
        "You are a compensation analyst. Use list_reports to get a manager's team, then "
        "for each report compare their salary to the band mid/min/max for their job_family "
        "and level (use get_comp_bands or the salary_band already returned). Flag anyone "
        "below band mid or outside the band. If a salary is NULL, note it cannot be assessed. "
        "Return a concise summary — do not open any cases."
    ),
    "tools": HR_TOOLS,
}

agent = create_deep_agent(
    model=model,
    tools=HR_TOOLS,
    system_prompt=(
        "You are an internal HR assistant and coordinator. For team-wide compensation "
        "reviews, delegate to the comp-analyst subagent using the task() tool, then "
        "summarize its findings for the user."
    ),
    subagents=[comp_analyst_subagent],
    checkpointer=MemorySaver(),
)
agent

In [ ]:
# Pick a manager who actually has reports (the summary's lonely manager is 1201).
config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Who reports to manager 1062, and are any of them below band mid?"}]
}, config=config)

def truncate(text, limit=1200):
    return text if len(text) <= limit else text[:limit] + "…"

for m in result["messages"]:
    m.model_copy(update={"content": truncate(m.text)}).pretty_print()

## 1.4 Backends & Memory

By default, files live in ephemeral state (`StateBackend`). Use `CompositeBackend` to route paths — e.g. `/memories/` to a persistent `StoreBackend` while everything else stays ephemeral.

`StoreBackend` is a database-backed store meant to persist memory across threads. Here we back it with a local SQLite file, but the LangGraph `Store` can be backed by the database of your choice (Postgres, etc.).

`StoreBackend` scopes what it reads and writes by `namespace` — a required argument, and your lever for isolation: per user, per assistant, or shared across everyone as here.

For an HR assistant, long-term memory is a natural fit for **policy** the agent should apply consistently across conversations — for example, the threshold your team uses to flag comp reviews.

In [ ]:
import sqlite3
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend
from langgraph.store.sqlite import SqliteStore

MEMORY_DB = "deep_agents_memory.db"

def open_memory_store(path=MEMORY_DB):
    """Open (or create) a persistent SQLite-backed long-term memory store."""
    conn = sqlite3.connect(path, check_same_thread=False, isolation_level=None)
    store = SqliteStore(conn)
    store.setup()  # creates tables IF NOT EXISTS -> safe whether or not the DB exists
    return store

store = open_memory_store()

# Pass a CompositeBackend *instance* (not a factory)
backend = CompositeBackend(
    default=StateBackend(),                                  # ephemeral scratch space
    routes={
        "/memories/": StoreBackend(                          # persists across threads (SQLite on disk)
            store=store,
            namespace=lambda rt: ("memories", "shared"),
        ),
    },
)

agent = create_deep_agent(
    model=model,
    tools=HR_TOOLS,
    system_prompt=(
        "You are an internal HR assistant. Save durable policy and preferences to "
        "/memories/ for future reference. ALWAYS check /memories files before answering "
        "so you apply the team's policies consistently."
    ),
    subagents=[comp_analyst_subagent],
    backend=backend,
    store=store,
    checkpointer=MemorySaver(),
)

# Thread 1: agent saves a policy to long-term memory
config1 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Our policy: flag any employee paid below band mid for a comp review. Save this to /memories/comp_policy.md"}]
}, config=config1)
print("Thread 1:", result["messages"][-1].text)

In [ ]:
# Thread 2: different thread, but /memories/ persists via StoreBackend
config2 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "What is our comp-review flagging policy? Check /memories/"}]
}, config=config2)
print("Thread 2:", result["messages"][-1].text)

## 1.5 Middleware: Pluggable Behavior

Middleware hooks into `wrap_model_call` (every LLM call) and `wrap_tool_call` (every tool call). This lets you inject rules, audit, or intercept without changing agent code.

For HR, this is where **confidentiality rules** and an **audit trail** belong — compensation data is sensitive, and every lookup/case action should be logged.

<img src="../images/deepAgentMiddleware.png" style="width: auto; max-height: 380px; border-radius: 8px;">

### Built-in context management

Three strategies the deep-agent middleware uses to keep within the model's context window:

<img src="../images/Offloading Inputs LangChain.png" style="width: auto; max-height: 340px; border-radius: 8px;">

**Offload Large Inputs** — file write/edit tool calls leave the full content in conversation history. At ~85% context capacity, deep agents truncate older tool calls and replace them with a file-pointer reference.

<img src="../images/Offloading Results LangChain.png" style="width: auto; max-height: 340px; border-radius: 8px;">

**Offload Large Results** — tool results over ~20k tokens are written to the backend and swapped with a path + 10-line preview. The agent can re-read or grep the full content as needed.

<img src="../images/LangChain Summarization.png" style="width: auto; max-height: 340px; border-radius: 8px;">

**Conversation Summarization** — when there's nothing left to offload and context hits ~85% of `max_input_tokens`, history is summarized. Full messages move to `/conversation_history/`; a structured summary replaces them in working memory.

In [ ]:
from langchain.agents.middleware import wrap_model_call, wrap_tool_call
from langchain_core.messages import SystemMessage
from datetime import datetime

audit_log = []

@wrap_model_call
def compliance_rules(request, handler):
    """Inject HR confidentiality rules into every LLM call."""
    rules = """## HR Confidentiality Rules
- Treat all compensation and personal data as confidential.
- Never disclose one employee's salary to a different employee.
- Do not reveal SSNs, bank details, or home addresses.
- When discussing pay, frame it against the band (min/mid/max), not as gossip."""
    existing = request.system_message
    blocks = list(existing.content_blocks) if existing else []
    blocks.append({"type": "text", "text": f"\n\n{rules}"})
    return handler(request.override(system_message=SystemMessage(content_blocks=blocks)))

@wrap_tool_call
def audit_trail(request, handler):
    """Create an audit log entry for every tool call (HR actions must be traceable)."""
    entry = {"tool": request.tool_call["name"], "timestamp": datetime.now().isoformat()}
    result = handler(request)
    entry["status"] = "success"
    audit_log.append(entry)
    return result

agent_with_middleware = create_deep_agent(
    model=model,
    tools=HR_TOOLS,
    system_prompt="You are an internal HR assistant.",
    middleware=[compliance_rules, audit_trail],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_middleware.invoke({
    "messages": [{"role": "user", "content": "What is the comp band for Software Engineering L3? And look up employee 1002."}]
}, config=config)

print(result["messages"][-1].text)
print(f"\n--- Audit Log ({len(audit_log)} entries) ---")
for entry in audit_log:
    print(f"  {entry['timestamp']}  {entry['tool']}  {entry['status']}")

## 1.6 HITL: Tool-Level Approval

Deep Agents supports `interrupt_on` — pause execution when specific tools are called. The human can approve, edit, or reject.

For a retrieve-then-act agent, the natural thing to gate is **the act step**. Reading data is safe; *opening an HR case* is a write to a system of record, so we require human approval before `open_hr_case` runs.

<img src="../images/deepAgentHITL.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
agent_with_hitl = create_deep_agent(
    model=model,
    tools=HR_TOOLS,
    system_prompt=hr_system_prompt,
    checkpointer=MemorySaver(),
    interrupt_on={
        "open_hr_case": True,   # gate the "act" step — writes to the cases table
    },
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_hitl.invoke({
    "messages": [{"role": "user", "content": "Employee 1001 is below band min. Open a comp review case for them."}]
}, config=config)

if result.get("__interrupt__"):
    interrupt_info = result["__interrupt__"][0].value
    for action in interrupt_info["action_requests"]:
        print(f"Paused — tool: {action['name']}, args: {action['args']}")
    print("\nWaiting for approval...")

In [ ]:
# Approve and continue — the case is written only after human sign-off.
if result.get("__interrupt__"):
    result = agent_with_hitl.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
    )
    print("Approved!")
    print("Agent reply:", result["messages"][-1].text)

In [ ]:
# Confirm the case landed in the cases table (the system of record).
import sqlite3
conn = sqlite3.connect(DB); conn.row_factory = sqlite3.Row
for r in conn.execute("SELECT case_id, employee_id, category, summary, status, created_at FROM cases ORDER BY case_id"):
    print(dict(r))
conn.close()

## 1.7 AGENTS.md & Skills

`AGENTS.md` replaces hardcoded system prompts with an editable identity file. Skills are loaded on demand — the agent reads them only when the task matches.

Here the agent's identity is the HR assistant, and we add a **comp-review-memo** skill that formats a manager-facing memo when someone is found outside band.

In [ ]:
from deepagents.backends.utils import create_file_data

agents_md = """# Internal HR Assistant

You are an internal HR assistant. You answer employee and manager questions by
querying the HR database, then take an action when asked. You are retrieve-then-act,
not a document search bot.

## Workflow
1. Plan multi-step requests with write_todos.
2. Retrieve facts with the tools (lookup_employee, list_reports, get_comp_bands).
   Never guess salaries, bands, or reporting lines.
3. Reason: to check "inside band", compare salary to band min/mid/max for the
   employee's job_family and level. If salary is NULL, say it can't be assessed.
4. Act only when asked: open_hr_case writes to the system of record.

## Rules
- Prefer employee_id over name; if a name is ambiguous, ask to disambiguate.
- Treat compensation data as confidential.
- When writing a comp-review memo, check /skills/ for the format.
"""

comp_memo_skill = """---
name: comp-review-memo
description: Format a manager-facing compensation-review memo. Use when asked to write up or summarize a comp review or band finding.
---

# Comp-Review Memo Skill

- Subject line: "Comp review: <employee name> (id <id>)"
- One-line summary: current salary vs. band (min/mid/max) and where it falls.
- Finding: state clearly whether they are below min, below mid, in range, or above max.
- Recommendation: a single suggested next step (e.g., open a band-adjustment case).
- Keep it under 120 words, neutral and factual. No gossip, no other employees' pay.
"""

# The agent's identity lives in /AGENTS.md (seeded below) and is loaded via the
# `memory` parameter — no redundant `system_prompt` string.
agent = create_deep_agent(
    model=model,
    tools=HR_TOOLS,
    subagents=[comp_analyst_subagent],
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": uuid7()}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Check whether employee 1001 is inside band, then write a comp-review memo about them. Do not call write_todos, just do it."}],
    "files": {
        "/AGENTS.md": create_file_data(agents_md),
        "/skills/comp-review-memo/SKILL.md": create_file_data(comp_memo_skill),
    },
}, config=config)

for m in result["messages"]:
    m.model_copy(update={"content": truncate(m.text)}).pretty_print()

## 1.8 The Complete Agent

All pieces together: tools, subagents, memory, middleware, HITL, AGENTS.md, and skills.

> `agents/hr_agent.py` packages a minimal slice of this (no HITL, no FilesystemBackend) for Module 3's traces/evals.

In [ ]:
store = open_memory_store()  # same SQLite-backed long-term memory as section 1.4
audit_log = []  # reset

complete_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/memories/": StoreBackend(
            store=store,
            namespace=lambda rt: ("memories", "shared"),
        ),
    },
)

complete_agent = create_deep_agent(
    model=model,
    tools=HR_TOOLS,
    subagents=[comp_analyst_subagent],
    backend=complete_backend,
    store=store,
    middleware=[compliance_rules, audit_trail],
    checkpointer=MemorySaver(),
    interrupt_on={"open_hr_case": True},   # gate the "act" step
    memory=["/AGENTS.md"],
    skills=["/skills/"],
)

print("Complete HR agent created with:")
print("  - Custom tools (lookup_employee, list_reports, get_comp_bands, open_hr_case)")
print("  - Subagent (comp-analyst)")
print("  - Memory (/memories/ -> StoreBackend on SQLite)")
print("  - Middleware (confidentiality rules + audit trail)")
print("  - HITL (approval before open_hr_case)")
print("  - AGENTS.md + Skills")

In [ ]:
# Drive the complete agent end-to-end.
# Exercises: subagent delegation, /memories/ policy, middleware (audit + confidentiality),
# and the HITL-gated "act" step (open_hr_case).
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": str(uuid7())}}

# Seed AGENTS.md + the comp-review-memo skill (same content as the 1.7 cell).
seed_files = {
    "/AGENTS.md": create_file_data(agents_md),
    "/skills/comp-review-memo/SKILL.md": create_file_data(comp_memo_skill),
}

result = complete_agent.invoke({
    "messages": [HumanMessage(content=(
        "Review manager 1062's team for anyone below band mid. For the most underpaid "
        "report, open a comp_review case, then save a one-line note to "
        "/memories/comp_reviews.md."
    ))],
    "files": seed_files,
}, config=config)

# The HITL middleware pauses on open_hr_case. Approve any pending actions.
while result.get("__interrupt__"):
    payload = result["__interrupt__"][0].value
    actions = payload.get("action_requests", [])
    for action in actions:
        print(f"  HITL pause -> approving {action['name']}: {action['args']}")
    result = complete_agent.invoke(
        Command(resume={"decisions": [{"type": "approve"} for _ in actions]}),
        config=config,
    )

print("\nFinal reply:\n", result["messages"][-1].text[:600])
print()
# Show only files the agent wrote (skip the seed files we passed in).
seed_paths = set(seed_files.keys())
agent_files = {k: v for k, v in (result.get("files") or {}).items() if k not in seed_paths}
print_files({"files": agent_files}, header="FILES THE AGENT WROTE")

print(f"\nAudit log: {len(audit_log)} tool call(s) recorded by the audit middleware")
for entry in audit_log:
    print(f"  {entry['timestamp']}  {entry['tool']:20s} {entry['status']}")

### Deep Agents Recap

| Feature | How | Built-in? |
|---------|-----|----------|
| **Harness** | `create_deep_agent()` | Filesystem, Planning, Summarization |
| **Custom tools** | `tools=[your_tool]` | Added to built-in tools |
| **Subagents** | `subagents=[{name, description, ...}]` | `task()` tool |
| **Memory** | `CompositeBackend` routing to `StoreBackend` | Path-based routing |
| **Middleware** | `middleware=[wrap_model_call, wrap_tool_call]` | Appended to built-in stack |
| **HITL** | `interrupt_on={"open_hr_case": True}` | Configurable per tool |
| **AGENTS.md** | `memory=["/AGENTS.md"]` or `files={}` | Editable identity |
| **Skills** | `skills=["./skills/"]` or `files={}` | On-demand capabilities |